# Third-review GPU run 2 on Kaggle (M3 noise sweep, M1 new models, X1 leftovers)

Runs `correction/run_review3b_kaggle.py`: the synthetic-noise sweep (M3), new models (M1: Qwen3 0.6B/4B/8B,
Gemma 3 1B/4B, Llama 3.2 3B) and the Grad-CAM maps missing from run 1 (X1). See `context/decisions.md` D41.

**Before running:**
1. Session options: **Accelerator = GPU T4 x2**, Internet = on.
2. Add data: the private dataset that holds `ocr_eval40.jsonl` (the same one as before).
   To resume a stopped run, also add that run's output (it contains a `review3b/` folder).
3. Add-ons, Secrets: tick `HF_TOKEN` for this notebook (Gemma, Gemma 3 and Llama are gated; accept the
   Gemma 3 and Llama 3.2 licences on Hugging Face first).
4. Set `HOURS` in the first code cell to your remaining GPU quota minus about 45 minutes (at most 11).
5. Save Version, then "Save & Run All (Commit)". It runs in the background; the browser can be closed.

When it finishes, download `review3b.zip` from the version's Output tab.


In [ ]:
import glob, os, shutil, subprocess, sys, time

# Hours of jobs: remaining Kaggle GPU quota minus about 45 minutes for setup and saving.
HOURS = 11.0

REPO_DIR = "/tmp/bengali-postocr-sllm"
OUT_DIR = "/kaggle/working/review3b"
os.makedirs(OUT_DIR, exist_ok=True)
SETUP_LOG = f"{OUT_DIR}/setup.log"

def sh(cmd):
    """Run a shell command, append its output to setup.log, return True on success."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    with open(SETUP_LOG, "a") as f:
        f.write(f"\n$ {cmd}\nexit {r.returncode}\n{r.stdout[-3000:]}{r.stderr[-3000:]}")
    print(("ok   " if r.returncode == 0 else "FAIL ") + cmd)
    return r.returncode == 0

if not os.path.isdir(REPO_DIR):
    sh(f"git clone -q https://github.com/Ariffurrhmn/bengali-postocr-sllm.git {REPO_DIR}")
assert os.path.isdir(REPO_DIR), "git clone failed: is Internet on in Session options? See review3/setup.log"
# transformers pinned to the version of the main runs (10-12 October 2026); if that
# version will not install on this image, fall back to the latest and record it.
if not sh("pip install -q transformers==5.19.0 accelerate sentencepiece jiwer protobuf"):
    sh("pip install -q -U transformers accelerate sentencepiece jiwer protobuf")

commit = subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout
import torch, transformers
env = (f"repo commit: {commit}python: {sys.version.split()[0]}\ntorch: {torch.__version__}\n"
       f"cuda: {torch.version.cuda}\ncudnn: {torch.backends.cudnn.version()}\n"
       f"transformers: {transformers.__version__}\ngpus: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}\n")
open(f"{OUT_DIR}/environment.txt", "w").write(env)
print(env)
assert torch.cuda.device_count() == 2, "Set Accelerator to GPU T4 x2"


In [ ]:
# Gemma, Gemma 3 and Llama are gated. Without the token only their jobs fail; the rest still run.
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Hugging Face login ok")
except Exception as e:
    print("WARNING: no Hugging Face login (Gemma and Llama jobs will fail):", e)


In [ ]:
ocr_files = glob.glob("/kaggle/input/**/ocr_eval40.jsonl", recursive=True)
assert ocr_files, "Attach the dataset that contains ocr_eval40.jsonl"
OCR = ocr_files[0]
print("OCR file:", OCR, "pages:", sum(1 for _ in open(OCR, encoding="utf-8")))


In [ ]:
# All jobs, in priority order; progress lines appear here, details in review3b/*.log.
subprocess.run([sys.executable, "run_review3b_kaggle.py", "--ocr-path", OCR, "--out-dir", OUT_DIR,
                "--hours", str(HOURS)], cwd=f"{REPO_DIR}/correction")


In [ ]:
import collections, json
for f in sorted(glob.glob(f"{OUT_DIR}/*.jsonl")):
    rows = [json.loads(l) for l in open(f, encoding="utf-8") if l.strip()]
    errs = sum(1 for r in rows if r.get("error"))
    keys = collections.Counter((r.get("model_key") or r.get("model"), r.get("engine")) for r in rows)
    print(os.path.basename(f), len(rows), "rows", f"errors {errs}" if errs else "", dict(keys))
shutil.make_archive("/kaggle/working/review3b", "zip", "/kaggle/working", "review3b")
print("wrote /kaggle/working/review3b.zip")
